<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">06 — Index the Vectors in Qdrant</div>

This notebook reads the embeddings produced by notebook 05 and stores them, with their metadata, in a **Qdrant** collection so they can be searched.

It is **Step 6** of the RAG data indexing pipeline — vector indexing only.

---

**What this notebook does:**
1. Connects to the Qdrant server and runs a health check
2. Loads the vectors (`05_embeddings.npy`) and aligned metadata (`05_embeddings_meta.jsonl`)
3. Creates (or resets) the Qdrant collection with the correct vector size and distance
4. Builds Qdrant points (vector + payload) using a stable UUID per chunk
5. Uploads the points in batches
6. Confirms the stored point count
7. Runs a sample vector search
8. Runs a metadata-filtered search
9. Prints the final collection status

**What this notebook intentionally does NOT do:**
- No answer generation, no LLM calls
- No reranking or hybrid retrieval

> **Before running:** Qdrant must be running. With Docker: `make up` starts it at
> `http://localhost:6333`. Inside the Jupyter container the host is `http://qdrant:6333`.

---
## 1. Imports

We need:
- **`json`**, **`os`**, **`uuid`** — read metadata, read env vars, build stable point ids
- **`pathlib.Path`** — OS-independent file paths
- **`numpy`** — load the vector matrix
- **`QdrantClient`** and **`models`** — connect to Qdrant and build collection / point objects

In [1]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
import os
import uuid
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient, models

print("Imports ready.")

Imports ready.


---
## 2. Configuration — Qdrant and Collection

| Setting | Meaning |
|---|---|
| `QDRANT_URL` | Where the Qdrant server is. Defaults to the Docker hostname, falls back to localhost |
| `COLLECTION` | Name of the collection to create / refresh |
| `DISTANCE` | Similarity metric. `Cosine` pairs with the normalized vectors from notebook 05 |
| `BATCH_SIZE` | How many points to upload per request |
| `UUID_NAMESPACE` | Fixed namespace so the same `chunk_id` always maps to the same point id |

`QDRANT_URL` and the collection name are read from environment variables when
present (so the same notebook works inside Docker and locally), with sensible
defaults otherwise.

In [2]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

# Read from env when available; default to the Docker service hostname
QDRANT_URL = os.environ.get("QDRANT_URL", "http://qdrant:6333")
COLLECTION = os.environ.get("QDRANT_COLLECTION", "rag_scifact")

DISTANCE   = models.Distance.COSINE
BATCH_SIZE = 256

# Fixed namespace -> deterministic point ids derived from each chunk_id
UUID_NAMESPACE = uuid.UUID("12345678-1234-5678-1234-567812345678")

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Inputs — produced by notebook 05
VECTORS_FILE = ROOT / "data" / "processed" / "05_embeddings.npy"
META_FILE    = ROOT / "data" / "processed" / "05_embeddings_meta.jsonl"

print(f"Qdrant URL   : {QDRANT_URL}")
print(f"Collection   : {COLLECTION}")
print(f"Vectors file : {VECTORS_FILE}  (exists: {VECTORS_FILE.exists()})")
print(f"Meta file    : {META_FILE}  (exists: {META_FILE.exists()})")

Qdrant URL   : http://qdrant:6333
Collection   : rag_scifact
Vectors file : /app/data/processed/05_embeddings.npy  (exists: True)
Meta file    : /app/data/processed/05_embeddings_meta.jsonl  (exists: True)


---
## 3. Connect to Qdrant and Health Check

We create the client and list the existing collections. If this cell fails, Qdrant
is probably not running — start it with `make up`.

In [3]:
# ── [3 / 10] Connect to Qdrant and Health Check ─────────────────────────────

try:
    client = QdrantClient(url=QDRANT_URL, timeout=30)
    existing = [c.name for c in client.get_collections().collections]
except Exception as exc:  # noqa: BLE001 - surface a friendly hint in the notebook
    raise ConnectionError(
        f"Could not reach Qdrant at {QDRANT_URL}.\n"
        "Start it with `make up` (Docker), then re-run this cell."
    ) from exc

print(f"Connected to Qdrant at {QDRANT_URL}")
print(f"Existing collections: {existing or '(none yet)'}")

Connected to Qdrant at http://qdrant:6333
Existing collections: (none yet)


---
## 4. Load Vectors and Metadata

We load the vector matrix and the aligned metadata, then confirm they have the
same number of rows (one metadata record per vector).

In [4]:
# ── [4 / 10] Load Vectors and Metadata ──────────────────────────────────────

if not VECTORS_FILE.exists() or not META_FILE.exists():
    raise FileNotFoundError(
        "Missing embedding files. Run notebook 05_generate_embeddings.ipynb first."
    )

vectors = np.load(VECTORS_FILE)

metadata = []
with META_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        metadata.append(json.loads(line))

vector_size = int(vectors.shape[1])

print(f"Vectors  : {vectors.shape}")
print(f"Metadata : {len(metadata):,} records")
print(f"Vector size (Qdrant dim): {vector_size}")

assert vectors.shape[0] == len(metadata), "vectors and metadata are misaligned"

Vectors  : (12281, 384)
Metadata : 12,281 records
Vector size (Qdrant dim): 384


---
## 5. Create (or Reset) the Collection

A Qdrant **collection** is like a table for vectors. We make this step idempotent:
if the collection already exists we delete it first, then create a fresh one. This
is exactly the "collection reset / index refresh" behavior the project goal asks for.

The collection is created with the vector size from the embeddings and Cosine
distance (which matches our normalized vectors).

In [5]:
# ── [5 / 10] Create (or Reset) the Collection ───────────────────────────────

if client.collection_exists(COLLECTION):
    print(f"Collection '{COLLECTION}' already exists — deleting it for a clean rebuild.")
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=vector_size, distance=DISTANCE),
)

print(f"Created collection '{COLLECTION}' (size={vector_size}, distance={DISTANCE.name}).")

Created collection 'rag_scifact' (size=384, distance=COSINE).


---
## 6. Build the Points

A Qdrant **point** is one stored item: an `id`, a `vector`, and a `payload`
(the metadata). Qdrant point ids must be an unsigned integer or a UUID, but our
`chunk_id` is a SHA-1 hex string. We deterministically convert each `chunk_id`
into a UUID with `uuid.uuid5`, so re-running keeps the same ids (no duplicates).
We also keep the original `chunk_id` inside the payload.

In [6]:
# ── [6 / 10] Build the Points ───────────────────────────────────────────────

def point_id_for(chunk_id: str) -> str:
    """Deterministically map a chunk_id to a Qdrant-compatible UUID string."""
    return str(uuid.uuid5(UUID_NAMESPACE, chunk_id))


points = []
for vector, meta in zip(vectors, metadata):
    points.append(
        models.PointStruct(
            id=point_id_for(meta["chunk_id"]),
            vector=vector.tolist(),
            payload=meta,  # store all metadata fields, including text
        )
    )

print(f"Built {len(points):,} points.")
print(f"Example point id : {points[0].id}")
print(f"Example payload keys : {list(points[0].payload.keys())}")

Built 12,281 points.
Example point id : 84841732-7bc9-50b6-9814-4b1a7baef57a
Example payload keys : ['chunk_id', 'document_id', 'chunk_index', 'source_file', 'document_type', 'title', 'created_at', 'text']


---
## 7. Upload the Points in Batches

We upsert the points in batches so we never send one giant request. `wait=True`
makes Qdrant confirm each batch is written before we continue.

In [7]:
# ── [7 / 10] Upload the Points in Batches ───────────────────────────────────

total = len(points)
uploaded = 0

for start in range(0, total, BATCH_SIZE):
    batch = points[start:start + BATCH_SIZE]
    client.upsert(collection_name=COLLECTION, points=batch, wait=True)
    uploaded += len(batch)
    print(f"  uploaded {uploaded:,} / {total:,}", end="\r")

print()
print(f"Upload complete: {uploaded:,} points.")

  uploaded 12,281 / 12,281
Upload complete: 12,281 points.


---
## 8. Sample Vector Search

To prove the index works, we take the vector of the first chunk and ask Qdrant for
its nearest neighbors. The top hit should be the chunk itself (score ≈ 1.0).

In [8]:
# ── [8 / 10] Sample Vector Search ───────────────────────────────────────────

query_vector = vectors[0].tolist()

results = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    limit=3,
    with_payload=True,
)

print("Top 3 nearest neighbors for chunk 0:")
for rank, hit in enumerate(results.points, start=1):
    title = (hit.payload.get("title") or "")[:70]
    print(f"  {rank}. score={hit.score:.4f}  doc={hit.payload.get('document_id')}  {title}")

Top 3 nearest neighbors for chunk 0:
  1. score=1.0000  doc=4983  Microstructural development of human newborn cerebral white matter ass
  2. score=0.9013  doc=4983  Microstructural development of human newborn cerebral white matter ass
  3. score=0.8962  doc=4983  Microstructural development of human newborn cerebral white matter ass


---
## 9. Metadata-Filtered Search

Qdrant can combine vector similarity with a metadata filter. Here we repeat the
search but restrict results to one `document_id`, showing how a retrieval service
could scope results by metadata.

In [9]:
# ── [9 / 10] Metadata-Filtered Search ───────────────────────────────────────

target_doc = metadata[0]["document_id"]

filtered = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    query_filter=models.Filter(
        must=[models.FieldCondition(
            key="document_id",
            match=models.MatchValue(value=target_doc),
        )]
    ),
    limit=5,
    with_payload=True,
)

print(f"Filtered search (document_id == {target_doc}):")
for rank, hit in enumerate(filtered.points, start=1):
    print(f"  {rank}. score={hit.score:.4f}  chunk_index={hit.payload.get('chunk_index')}")

# Every returned point must belong to the requested document
assert all(h.payload["document_id"] == target_doc for h in filtered.points), \
    "filter returned a chunk from another document"
print("\nFilter verified — all results belong to the requested document.")

Filtered search (document_id == 4983):
  1. score=1.0000  chunk_index=0
  2. score=0.9013  chunk_index=2
  3. score=0.8962  chunk_index=1

Filter verified — all results belong to the requested document.


---
## 10. Verify the Collection Status

Finally we read back the collection info and the exact point count, and confirm it
equals the number of vectors we uploaded.

In [10]:
# ── [10 / 10] Verify the Collection Status ──────────────────────────────────

count = client.count(collection_name=COLLECTION, exact=True).count
info  = client.get_collection(COLLECTION)

print("=" * 60)
print(f"Collection : {COLLECTION}")
print("=" * 60)
print(f"  Points stored : {count:,}")
print(f"  Vectors built : {len(points):,}")
print(f"  Status        : {info.status}")
print()

assert count == len(points), "stored point count does not match uploaded points"
print("Indexing verified — every chunk is stored and searchable in Qdrant.")
print()
print("Next: run 07_index_health_report.ipynb for a full end-to-end query demo.")

Collection : rag_scifact
  Points stored : 12,281
  Vectors built : 12,281
  Status        : green

Indexing verified — every chunk is stored and searchable in Qdrant.

Next: run 07_index_health_report.ipynb for a full end-to-end query demo.
